# Used Car Depreciation Analysis

## Data Preparation

Explore, prepare, validate, and transform used vehicle
depreciation data to compare value retention across
four powertrain categories: Gasoline, HEV, PHEV, and BEV.


## Import Libraries

Import Pandas for data exploration, manipulation,
and transformation.


## Load Dataset

Load the depreciation dataset from US Vehicle Trends.
Preserve the original dataset in the data/raw directory.

[Original Dataset](https://github.com/vehicletrends/vehicletrends/blob/main/data-raw/depreciation.csv)


## Data Credibility Assessment (ROCCC)

### Reliable

The dataset is produced by researchers using licensed
vehicle listing data from MarketCheck.

The broader research project provides nationwide
coverage and documents its depreciation methodology.

However, the underlying listing records are proprietary,
which limits independent verification. Listing prices
may also differ from actual transaction prices.

Assessment: Reasonably reliable for exploratory
depreciation analysis, subject to these limitations.


### Original

The underlying vehicle listing data originates from
MarketCheck and is licensed by US Vehicle Trends.

US Vehicle Trends processes and aggregates the original
listing data into publicly available summary datasets.

Assessment: The data has a traceable origin. However,
the original listing records are proprietary and cannot
be independently accessed or verified.


### Comprehensive

The dataset provides nationwide coverage across all four
powertrain categories required for this analysis.

It contains value retention ratios across multiple
vehicle age intervals, allowing consistent comparisons
of depreciation across powertrains.

However, the dataset does not include calendar-year
breakdowns, actual sales volume, dealership profitability,
inventory turnover, or underlying sample sizes per group.

Assessment: Sufficiently comprehensive for age-based
depreciation analysis, but additional data sources are
required to address the broader business objectives.


### Current

The US Vehicle Trends project covers vehicle listing
data from 2018 through 2025.

However, the depreciation dataset aggregates information
by vehicle age rather than calendar year. Therefore,
recent changes in depreciation cannot be isolated
from this dataset alone.

Assessment: Sufficiently recent for general age-based
depreciation analysis, but limited in its ability to
represent current market conditions.


### Cited

US Vehicle Trends identifies its original data provider,
research contributors, and calculation methodology.

The project provides dataset documentation, data
dictionaries, a public GitHub repository, and official
citation guidance.

Assessment: The dataset is appropriately documented
and attributable to identifiable sources.

#### References

- [US Vehicle Trends](https://vehicletrends.us/about)
- [Dataset Documentation](https://pkg.vehicletrends.us/articles/datasets)
- [GitHub Repository](https://github.com/vehicletrends/vehicletrends)


## Data Licensing and Usage

US Vehicle Trends publishes aggregated vehicle
listing summaries for public access.

The website and its content are licensed under
CC BY-SA 4.0, while the associated GitHub package
repository uses the MIT license.

The underlying MarketCheck records are proprietary.

The processed CSV will remain local until its
redistribution permissions are confirmed.
All analysis will include appropriate source attribution.


In [ ]:
import pandas as pd

df_depreciation = pd.read_csv("../data/raw/depreciation.csv")

## Initial Data Exploration

Examine the dataset structure, column names, data types,
missing values, and descriptive statistics.


In [ ]:
df_depreciation.head()

In [ ]:
df_depreciation.info()

## Explore Unique Values

Identify available powertrain categories, vehicle types,
quantiles, and vehicle age ranges to determine which
records are relevant to the project scope.


In [ ]:
df_depreciation["powertrain"].unique()

In [ ]:
df_depreciation["vehicle_type"].unique()

In [ ]:
df_depreciation["quantile"].unique()

In [ ]:
print("Max num from age_bin = ", df_depreciation["age_bin"].max())

In [ ]:
print("Min num from age_bin = ", df_depreciation["age_bin"].min())

In [ ]:
df_depreciation["rr"].describe()

## Investigate Potential Outliers

Investigate value retention ratios greater than 1.0,
which represent values above the original price benchmark.

Identify their distribution across quantiles before
determining whether further investigation is necessary.

Retain these records until there is sufficient evidence
to justify their removal.


In [ ]:
mask = df_depreciation["rr"] > 1.0
df_depreciation[mask]

In [ ]:
df_depreciation[mask]["quantile"].value_counts()

In [ ]:
mask = (df_depreciation["rr"] > 1.0) & (df_depreciation["quantile"] == 50)
df_depreciation[mask]

## Filter Dataset

Filter the dataset according to the project scope:

- Powertrain: Gasoline, HEV, PHEV, and BEV.
- Vehicle type: All vehicle categories combined.
- Quantile: 50th percentile (median).

Create an independent copy of the filtered dataset
to preserve the original data.

The filtered dataset contains 226 records.


In [ ]:
mask = (
    (
        df_depreciation["powertrain"].isin(
            [
                # Select the four powertrains within our project scope
                "Gasoline",
                "Hybrid Electric (HEV)",
                "Plug-In Hybrid Electric (PHEV)",
                "Battery Electric (BEV)",
            ]
        )
    )
    & (df_depreciation["vehicle_type"] == "All")
    & (df_depreciation["quantile"] == 50)
)

# Create an independent copy to avoid unintended changes to the original dataset
df_depreciation_filtered = df_depreciation[mask].copy()

df_depreciation_filtered.head()

In [ ]:
df_depreciation_filtered.shape

In [ ]:
df_depreciation_filtered.value_counts("powertrain")
# This is the df method. You can also use the Series method: df_depreciation_filtered['powertrain'].value_counts()

## Standardize Age Range

Examine the minimum and maximum vehicle ages for
each powertrain category.

Gasoline and HEV have a maximum age of 181.5 months,
while BEV and PHEV have a maximum age of 178.5 months.

Restrict all four powertrains to the shared age range
of 13.5 to 178.5 months to ensure consistent comparisons.

The standardized dataset contains 224 records.


In [ ]:
df_depreciation_filtered.groupby("powertrain")["age_bin"].agg(["min", "max"])

In [ ]:
mask = df_depreciation_filtered["age_bin"] <= 178.5

df_depreciation_filtered = df_depreciation_filtered[mask].copy()

df_depreciation_filtered.groupby("powertrain")["age_bin"].agg(["min", "max"])

In [ ]:
df_depreciation_filtered.shape

## Validate Filtered Dataset

Verify that each age interval contains exactly four
distinct powertrain categories.

### Validation Results

- Total records: 224
- Total age intervals: 56
- Distinct powertrains per interval: 4
- Duplicate records within each interval: 0

The dataset has consistent age coverage across
all four selected powertrains.


In [ ]:
df_depreciation_filtered.groupby("age_bin")["powertrain"].count()

In [ ]:
df_depreciation_filtered.groupby("age_bin")["powertrain"].nunique()

In [ ]:
# This verifies the number of distinct powertrains per age interval
df_depreciation_filtered.groupby("age_bin")["powertrain"].nunique().value_counts()

## Data Transformation

Create calculated columns to improve the interpretability
of the dataset and support subsequent analysis and
visualization.

- age_years: Vehicle age converted from months to years.
- retention_pct: Value retention expressed as a percentage.
- depreciation_pct: Vehicle depreciation expressed as a percentage.

Preserve the original columns for reference.


In [ ]:
df_depreciation_filtered["age_year"] = df_depreciation_filtered["age_bin"] / 12

df_depreciation_filtered.head()

In [ ]:
df_depreciation_filtered["retention_pct"] = df_depreciation_filtered["rr"] * 100

df_depreciation_filtered.head()

In [ ]:
df_depreciation_filtered["depreciation_pct"] = (
    100 - df_depreciation_filtered["retention_pct"]
)

df_depreciation_filtered.head()

### Validate Transformed Dataset

Verify the dataset after creating the calculated columns.

- Confirm the expected dimensions of 224 rows and 8 columns.
- Verify that all calculated columns have numeric data types.
- Confirm that no missing values were introduced.
- Ensure that the original record count is preserved.


In [ ]:
df_depreciation_filtered.shape

In [ ]:
df_depreciation_filtered.info()

### Reset DataFrame Index

Reset the DataFrame index after filtering and
transformation to create a sequential index
from 0 to 223.

Discard the original row indices to avoid creating
an unnecessary column.


In [ ]:
df_depreciation_filtered = df_depreciation_filtered.reset_index(drop=True)

df_depreciation_filtered.head()

### Transformation Results

Successfully created three calculated columns:

- age_year: Vehicle age in years.
- retention_pct: Value retention percentage.
- depreciation_pct: Depreciation percentage.

Validation confirmed:

- 224 records and 8 columns.
- No missing values across all columns.
- All calculated columns have the float64 data type.
- Original columns and record count are preserved
  during transformation.


## Export Processed Dataset

Export the cleaned, validated, and transformed dataset
to the data/processed directory for subsequent analysis.

The processed dataset contains 224 records and 8 columns.

Exclude the DataFrame index from the exported CSV file
to avoid introducing an unnecessary column.

Preserve the original dataset in the data/raw directory.


In [ ]:
df_depreciation_filtered.to_csv(
    "../data/processed/depreciation_processed.csv", index=False
)